# Simple tool to read the demo JSON data model into a Pydantic Model

In [1]:
import os
import json
import pandas as pd
import numpy as np
import requests
from types import List, Dict

from src.iea43_wra_data_model import IeaWindResourceAssessmentDataModel, MeasurementPointItem, MeasurementType

ImportError: cannot import name 'List' from 'types' (C:\Users\SPeters\.pyenv\pyenv-win\versions\3.11.9\Lib\types.py)

Note: Requires Pandas version +1.0

### Load the JSON file

In [9]:
# first try to load the file from your computer
cwd = os.getcwd()
file_path = os.path.join(cwd,"..", "demo_data", "iea43_wra_data_model.json")
with open(file_path) as json_file:
    meta_data = json.load(json_file)
print("Loaded JSON file from local machine.")

Loaded JSON file from local machine.


### Create Pydantic Model

In [13]:
iea_wra_data_model = IeaWindResourceAssessmentDataModel(**meta_data)
print("Created IeaWindResourceAssessmentDataModel object.")

Created IeaWindResourceAssessmentDataModel object.


### Show high level Data Model information

In [14]:
print(f'Author name:\t\t\t{iea_wra_data_model.author}')
print(f'Organisation:\t\t\t{iea_wra_data_model.organisation}')
print(f'Date Created:\t\t\t{iea_wra_data_model.date_created}')
print(f'IEA Data Model version:\t\t{iea_wra_data_model.version}')

print(f'Plant name:\t\t\t{iea_wra_data_model.plant_name}')
print(f'Plant type:\t\t\t{iea_wra_data_model.plant_type}')

Author name:			Stephen Holleran
Organisation:			brightwind
Date Created:			2021-12-23
IEA Data Model version:		1.3.0-2024.03
Plant name:			A Name of the Wind Farm
Plant type:			PlantType.onshore_wind


### Show all the measurement locations

In [17]:
# print a table of the meas_locs parameters.
meas_locs = []
for meas_loc in iea_wra_data_model.measurement_location:
    meas_locs.append(meas_loc.model_dump())
meas_locs_df = pd.DataFrame(meas_locs)
meas_locs_df.set_index('name', inplace=True)
display(meas_locs_df)

,uuid,latitude_ddeg,longitude_ddeg,measurement_station_type_id,notes,update_at,mast_properties,vertical_profiler_properties,logger_main_config,model_config_,measurement_point
name,,,,,,,,,,,
Test_MM1,6858cf5c-24e0-40d4-955b-8aecbccba391,53.5,-8.0,MeasurementStationTypeId.mast,I can write anything I want here.,2020-04-18 18:13:00,{'mast_geometry_id': MastGeometryId.lattice_tr...,None,"[{'logger_oem_id': LoggerOemId.NRG_Systems, 'l...",None,"[{'name': 'Spd_80.1_315', 'measurement_type_id..."


### Logger main configurations

In [22]:
logger_main_config = []
for meas_loc in iea_wra_data_model.measurement_location:
    for log_config in meas_loc.logger_main_config:
        logger_main_config.append(log_config.model_dump())

logger_main_config_df = pd.DataFrame(logger_main_config)
display(logger_main_config_df.set_index('logger_name'))

,logger_oem_id,logger_model_name,logger_serial_number,logger_firmware_version,logger_id,date_from,date_to,encryption_pin_or_key,enclosure_lock_details,data_transfer_details,offset_from_utc_hrs,sampling_rate_sec,averaging_period_minutes,timestamp_is_end_of_period,clock_is_auto_synced,logger_acquisition_uncertainty,uncertainty_k_factor,notes,update_at,lidar_config
logger_name,,,,,,,,,,,,,,,,,,,,
AName_MM1,LoggerOemId.NRG_Systems,Symphonie Plus3,01002,3.2.3,4321,2020-04-12 12:00:00,None,9876,combination lock PIN 54321,Emails to data@developername.com,-5.0,3,10,False,True,0.1,2.0,I can write anything I want here.,2020-04-18 18:13:00,None


In [ ]:
measurement_columns_map = []

for meas_loc in iea_wra_data_model.measurement_location:
    for measure_point in meas_loc.measurement_point:
        height_m = measure_point.height_m
        if measure_point.measurement_type_id == MeasurementType.wind_speed:
            perameter = "hws"
        elif measure_point.measurement_type_id == MeasurementType.wind_direction:
            perameter = "hwd"
        elif measure_point.measurement_type_id == MeasurementType.temperature:
            perameter = "tmp"
        elif measure_point.measurement_type_id == MeasurementType.pressure:
            perameter = "prs"
        elif measure_point.measurement_type_id == MeasurementType.relative_humidity:
            perameter = "rh"
        else:
            continue
        config = measure_point.logger_measurement_config[0]
        for column in config.columns:
            if column.statistic_type_id == "avg":
                statistic = "mean"
            elif column.statistic_type_id == "sd":
                statistic = "std"
            elif column.statistic_type_id == "min":
                statistic = "min"
            elif column.statistic_type_id == "max":
                statistic = "max"
            else:
                continue
            measurement_columns_map.append({
                column.column_name: f"{height_m:0.2f}m_{perameter}_{statistic}"           })

df = pd.DataFrame(measurement_columns_map)
df



MeasurementType.wind_speed


### Some functions to pull out and format the measurement points set ups.

In [ ]:
def _flatten_sensor_dict(sensor):
    """
        Flatten the sensor dictionary retrieved from jason
        assigning all the sub-dictionaries to the main dictionary.

        :param sensor: The sensor dictionary retrieved for a single configuration
                           option and meas_point id.
        :type sensor: dict
        :return: output
        :rtype: dict

    """
    output = {key: value for key, value in sensor.items() if (type(value) != list) or (value == {})}
    for key, value in zip(sensor.keys(), sensor.values()):
        if (type(value) == list):
            if key == 'calibration':
                value = {key + "_" + k: v for k, v in value[0].items()}
            output.update(value)
    return output

In [ ]:
def rename_variables(input_dict, root_name):
    for var_to_rename in ['height_m', 'serial_number', 'update_at', 'notes']:
        if var_to_rename in list(input_dict.keys()):
            input_dict[ root_name + '_' + var_to_rename] = input_dict.pop(var_to_rename)
    return input_dict

In [ ]:
def replace_none_date(input_dict):
    for date_str in ['date_from', 'date_to']:
        if input_dict[date_str] is None:
            input_dict[date_str] = '2100-12-31T00:00:00'
    return input_dict

In [ ]:
def get_meas_points(meas_points: List[MeasurementPointItem]) -> List[Dict]:
    meas_points_flatten = []
    for meas_point in meas_points:
#         meas_point = _flatten_meas_point_dict(meas_point)
        log_meas_configs = sorted( meas_point.logger_measurement_config, key=lambda i: i.date_from)
        log_meas_configs = [replace_none_date(rename_variables(log_meas_config, 'log_meas_config')) for log_meas_config in log_meas_configs]
        sensors = [replace_none_date(rename_variables(_flatten_sensor_dict(sensor), 'sensor')) for sensor in meas_point['sensor']]
        if meas_point['mounting_arrangement'] is not None:
            mounting_arrangements = [replace_none_date(rename_variables(mntg_arrang, 'mounting_arrangement')) 
                                 for mntg_arrang in meas_point['mounting_arrangement']]
        else:
            mounting_arrangements = {}
        
        date_from = [log_meas_config['date_from'] for log_meas_config in log_meas_configs]
        date_to = [log_meas_config['date_to'] for log_meas_config in log_meas_configs]
        for sensor in sensors:
            date_from.append(sensor['date_from'])
            date_to.append(sensor['date_to'])
        for mntg_arrang in mounting_arrangements:
            date_from.append(mntg_arrang['date_from'])
            date_to.append(mntg_arrang['date_to'])
        
        date_from.extend(date_to)
        dates = np.unique(date_from)
        for i in range(len(dates)-1): 
            good_log_meas_config = {}
            for log_meas_config in log_meas_configs:
                if (log_meas_config['date_from'] <= dates[i]) & (log_meas_config['date_to'] > dates[i]):
                    good_log_meas_config = log_meas_config.copy()
            if good_log_meas_config != {}:
                for sensor in sensors: 
                    if (sensor['date_from'] <= dates[i]) & (sensor['date_to'] > dates[i]) :
                        good_log_meas_config.update(sensor)
                for mntg_arrang in mounting_arrangements:
                    if (mntg_arrang['date_from'] <= dates[i]) & (mntg_arrang['date_to'] > dates[i]) :
                        good_log_meas_config.update(mntg_arrang)
                good_log_meas_config['date_to'] = dates[i+1]
                good_log_meas_config['date_from'] = dates[i]
                good_log_meas_config.update(meas_point)
                del good_log_meas_config['logger_measurement_config']
                del good_log_meas_config['sensor'] 
                meas_points_flatten.append(good_log_meas_config)
    return meas_points_flatten 

In [ ]:
def _format_sensor_table(meas_points, table_type='full'):
    
    if table_type == 'full':
        header = ['name', 'measurement_units', 'oem',
                  'height_m', 'boom_orientation_deg', 'vane_dead_band_orientation_deg',
                  'date_from', 'date_to', 'connection_channel', 'log_meas_config_height_m', 'slope', 'offset', 'calibration_slope',
                  'calibration_offset']
        header_for_report = ['Instrument Name', 'Units', 'Sensor OEM',
                        'Height [m]', 'Boom Orient. [deg, mag N]', 'Dead Band Orient. [deg, mag N]',
                        'Date From', 'Date To', 'Logger Channel', 'Logger Stated Height [m]', 'Logger Slope', 'Logger Offset', 'Calibration Slope',
                        'Calibration Offset']
    elif table_type == 'meas_points':
        header = ['name', 'measurement_type_id', 'height_m', 'boom_orientation_deg']
        header_for_report = ['Instrument Name', 'Measurement Type', 'Height [m]', 'Boom Orient. [deg, mag N]']    
    elif table_type == 'speed_info':
        header = ['name', 'measurement_units', 'oem', 'model', 'sensor_serial_number',
                  'height_m', 'boom_orientation_deg', 
                  'date_from', 'date_to', 'slope', 'offset', 'calibration_slope',
                  'calibration_offset', 'measurement_type_id']
        header_for_report = ['Instrument Name', 'Units', 'Sensor Make', 'Sensor Model', 'Serial No',
                             'Height [m]', 'Boom Orient. [deg, mag N]',
                             'Date From', 'Date To', 'Logger Slope', 'Logger Offset', 'Calibration Slope',
                             'Calibration Offset', 'measurement_type_id']
    elif table_type == 'direction_info':
        header = ['name', 'measurement_units', 'oem', 'model', 'sensor_serial_number',
                  'height_m', 'boom_orientation_deg', 'vane_dead_band_orientation_deg', 
                  'date_from', 'date_to', 'offset', 'measurement_type_id']
        header_for_report = ['Instrument Name', 'Units', 'Sensor Make', 'Sensor Model', 'Serial No',
                             'Height [m]', 'Boom Orient. [deg, mag N]', 'Dead Band Orient. [deg, mag N]',
                             'Date From', 'Date To', 'Logger Offset', 'measurement_type_id']
    
    sensors_table_report = pd.DataFrame(meas_points)

    if any(elem not in sensors_table_report.columns for elem in header):
        ind_to_remove = [ind for ind, elem in enumerate(header) if elem not in sensors_table_report.columns]
        del header[ind_to_remove[0]]
        del header_for_report[ind_to_remove[0]]
    
    sensors_table_report = pd.DataFrame(sensors_table_report[header])
    if table_type == 'speed_info':
        sensors_table_report = sensors_table_report[sensors_table_report['measurement_type_id'] == 'wind_speed']
        del sensors_table_report['measurement_type_id']
    if table_type == 'direction_info':
        sensors_table_report = sensors_table_report[sensors_table_report['measurement_type_id'] == 'wind_direction']
        del sensors_table_report['measurement_type_id']
    
    if 'date_from' in sensors_table_report.columns:
        sensors_table_report['date_from'] = pd.to_datetime(sensors_table_report['date_from'].values.astype(str), 
                                                           format='%Y-%m-%dT%H:%M:%S').strftime("%d-%b-%Y")
    if 'date_to' in sensors_table_report.columns:
        sensors_table_report['date_to'] = pd.to_datetime(sensors_table_report['date_to'].values.astype(str), 
                                                         format='%Y-%m-%dT%H:%M:%S').strftime("%d-%b-%Y")

    sensors_table_report = sensors_table_report.replace({np.nan: '-', 'NaT': '-', '31-Dec-2100':'-'})
    sensors_table_report.rename(columns={k: h for k, h in zip(header, header_for_report)}, inplace=True)
    index_name = 'Instrument Name'
    sensors_table_report = sensors_table_report.set_index(index_name)
    
    return sensors_table_report

### The main measurment points

In [ ]:
for meas_loc in meta_data['measurement_location']:  
    logger_meas_configs = get_meas_points(meas_loc['measurement_point'])
    sensors_table = _format_sensor_table(logger_meas_configs, table_type='meas_points')
    display(sensors_table.drop_duplicates())

,Measurement Type,Height [m],"Boom Orient. [deg, mag N]"
Instrument Name,,,
Spd_80.1_315,wind_speed,80.1,315.0
Spd_80mSE,wind_speed,80.2,135.0
Spd_60mNW,wind_speed,60.1,315.0
Spd_60mSE,wind_speed,60.2,135.0
Spd_40mNW,wind_speed,40.1,315.0
Spd_30mNW,wind_speed,30.1,315.0
Spd_40mSE,wind_speed,40.2,135.0
Dir_76mNW,wind_direction,76.1,315.0
Dir_56mNW,wind_direction,56.1,315.0


### More details on each measurement point

In [ ]:
for meas_loc in iea_wra_data_model.measurement_location:
    logger_meas_configs = get_meas_points(meas_loc.measurement_point)
    sensors_table = _format_sensor_table(logger_meas_configs)
    display(sensors_table)

,Sensor OEM,Height [m],"Boom Orient. [deg, mag N]","Dead Band Orient. [deg, mag N]",Date From,Date To,Logger Channel,Logger Stated Height [m],Logger Slope,Logger Offset,Calibration Slope,Calibration Offset
Instrument Name,,,,,,,,,,,,
Spd_80.1_315,Thies,80.1,315.0,-,12-Apr-2020,15-Apr-2020,CH1,80,0.04573,0.2419,0.04573,0.2419
Spd_80.1_315,Thies,80.1,315.0,-,15-Apr-2020,-,CH1,80,0.04573,0.2491,0.04573,0.2419
Spd_80mSE,Thies,80.2,135.0,-,12-Apr-2020,18-Apr-2020,CH2,80,0.04568,0.2487,0.04568,0.2487
Spd_80mSE,Thies,80.2,135.0,-,18-Apr-2020,-,CH2,80,0.04575,0.2497,0.04575,0.2497
Spd_60mNW,Thies,60.1,315.0,-,12-Apr-2020,-,CH3,60,0.04666,0.2416,0.04666,0.2416
Spd_60mSE,Thies,60.2,135.0,-,12-Apr-2020,-,CH4,60,0.04777,0.2417,0.04777,0.2417
Spd_40mNW,Thies,40.1,315.0,-,12-Apr-2020,18-Apr-2020,CH5,60,0.04888,0.2418,0.04888,0.2418
Spd_40mNW,Thies,40.1,315.0,-,18-Apr-2020,-,CH14,40,0.04888,0.2418,0.04888,0.2418
Spd_30mNW,Thies,30.1,315.0,-,12-Apr-2020,18-Apr-2020,CH6,30,0.04999,0.2419,0.04999,0.2419


### Anemometer (or wind speed) specific table

In [ ]:
for meas_loc in meta_data['measurement_location']:    
    sensors_table = _format_sensor_table(logger_meas_configs, table_type='speed_info')
    display(sensors_table.drop_duplicates())

,Sensor Make,Sensor Model,Serial No,Height [m],"Boom Orient. [deg, mag N]",Date From,Date To,Logger Slope,Logger Offset,Calibration Slope,Calibration Offset
Instrument Name,,,,,,,,,,,
Spd_80.1_315,Thies,4.3351.10.000,09183000,80.1,315.0,12-Apr-2020,15-Apr-2020,0.04573,0.2419,0.04573,0.2419
Spd_80.1_315,Thies,4.3351.10.000,09183000,80.1,315.0,15-Apr-2020,-,0.04573,0.2491,0.04573,0.2419
Spd_80mSE,Thies,4.3351.10.000,09183001,80.2,135.0,12-Apr-2020,18-Apr-2020,0.04568,0.2487,0.04568,0.2487
Spd_80mSE,Thies,4.3351.10.000,09183023,80.2,135.0,18-Apr-2020,-,0.04575,0.2497,0.04575,0.2497
Spd_60mNW,Thies,4.3351.10.000,09183002,60.1,315.0,12-Apr-2020,-,0.04666,0.2416,0.04666,0.2416
Spd_60mSE,Thies,4.3351.10.000,09183003,60.2,135.0,12-Apr-2020,-,0.04777,0.2417,0.04777,0.2417
Spd_40mNW,Thies,4.3351.10.000,09183004,40.1,315.0,12-Apr-2020,18-Apr-2020,0.04888,0.2418,0.04888,0.2418
Spd_40mNW,Thies,4.3351.10.000,09183004,40.1,315.0,18-Apr-2020,-,0.04888,0.2418,0.04888,0.2418
Spd_30mNW,Thies,4.3351.10.000,09183005,30.1,315.0,12-Apr-2020,18-Apr-2020,0.04999,0.2419,0.04999,0.2419


### Wind Vane (or wind direction) specific table

In [ ]:
for meas_loc in meta_data['measurement_location']:    
    sensors_table = _format_sensor_table(logger_meas_configs, table_type='direction_info')
    display(sensors_table.drop_duplicates())

,Sensor Make,Sensor Model,Serial No,Height [m],"Boom Orient. [deg, mag N]","Dead Band Orient. [deg, mag N]",Date From,Date To,Logger Offset
Instrument Name,,,,,,,,,
Dir_76mNW,NRG,#200P,01234589,76.1,315.0,315.0,12-Apr-2020,-,-
Dir_56mNW,NRG,#200P,01234567,56.1,315.0,315.0,12-Apr-2020,18-Apr-2020,-
Dir_56mNW,NRG,#200P,01234588,56.1,315.0,135.0,18-Apr-2020,-,-
